## Read data ftom team statistics

This notebook handles the ingestion of data from a s3 bucket and stores the data in a bronze layer

In [ ]:
import dlt
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


pass parameters to bundle later

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
source_path = dbutils.widgets.get("source_path")
file_format = "json"

In [ ]:
@dlt.table(
    name="teams_statistics",
    comment="Ingested teams data from JSON files with ingestion timestamp",
    table_properties={
        "quality": "bronze",
        "pipeline": "soccer_api_consumer"
    }
)
def ingest_teams_statistics():
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{source_path}/schemas/teams_statistics/") #auto schema inference location
        .load(f"{source_path}/teams_statistics/")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    return df

add more tables to the definition so it can read players

In [ ]:
@dlt.table(
    name="players_statistics",
    comment="Ingested players data from parquet files with ingestion timestamp",
    table_properties={
        "quality": "bronze",
        "pipeline": "soccer_api_consumer"
    }
)
def ingest_players_statistics():
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{source_path}/schemas/players/") #auto schema inference location
        .load(f"{source_path}/players/")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    return df